In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd
import numpy as np
import csv
import pickle
import copy
import re
import random
import matplotlib.pyplot as plt
import itertools
import json
import openai
import time


In [5]:
from google.colab import userdata
my_secret_key = userdata.get('API_KEY')

if my_secret_key:
  print("Token retrieved successfully.")
else:
  print("Token not found in Colab Secrets.")

from openai import OpenAI

client = OpenAI(
    # This is the default and can be omitted
    api_key = my_secret_key,
)


import asyncio
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key = my_secret_key,
)  # make sure this is your actual key



Token retrieved successfully.


In [ ]:
from sentence_transformers import SentenceTransformer

# Load new top-tier model
model = SentenceTransformer("intfloat/e5-large-v2")
# OR
# modfrom sentence_transformers import SentenceTransformer


print("Model loaded!")

In [11]:
!rm -rf LLM4BEAR
!git clone --depth 1 --filter=blob:none --sparse https://github.com/anon5159753/LLM4BEAR.git
!cd LLM4BEAR && git sparse-checkout set "BundleRec Data"
!cd LLM4BEAR && git sparse-checkout add "2_Bundle Refinement"



Cloning into 'LLM4BEAR'...
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 21 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (21/21), 8.12 KiB | 8.12 MiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 67 bytes | 67.00 KiB/s, done.
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 82 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 45.90 MiB | 8.87 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (83/83), done.
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 24

In [ ]:

import os

path = "/content/drive/MyDrive/Bundle_Refinement/"

try:
    os.makedirs(path, exist_ok=True)
    print(f"Successfully created: {path}")
except Exception as e:
    print(f"An error occurred: {e}")

In [7]:
clothing_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/clothing/bundle_list_items.pkl")

electronic_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/electronic/bundle_list_items.pkl")

food_bundles_items = pd.read_pickle("/content/LLM4BEAR/BundleRec Data/food/bundle_list_items.pkl")

bundle_items_list = [clothing_bundles_items, electronic_bundles_items, food_bundles_items]

clothing_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/bundle_intent.csv")

electronics_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/bundle_intent.csv")

food_intent = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/bundle_intent.csv")

intents = [clothing_intent, electronics_intent, food_intent]

clothing_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/merged_metadata.csv")

electronic_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/merged_metadata.csv")

food_metadata = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/merged_metadata.csv")

metadata = [clothing_metadata, electronic_metadata, food_metadata]

clothing_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_session.csv")

clothing_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_item.csv")

clothing_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/user_bundle.csv")

clothing_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_bundle.csv")

clothing_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/session_item.csv")

clothing_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/clothing/item_titles.csv")

electronic_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_session.csv")

electronic_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_item.csv")

electronic_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/user_bundle.csv")

electronic_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_bundle.csv")

electronic_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/session_item.csv")

electronic_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/electronic/item_titles.csv")

food_user_session = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_session.csv")

food_user_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_item.csv")

food_user_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/user_bundle.csv")

food_session_bundle = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_bundle.csv")

food_session_item = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/session_item.csv")

food_item_names = pd.read_csv("/content/LLM4BEAR/BundleRec Data/food/item_titles.csv")

user_session = [clothing_user_session, electronic_user_session, food_user_session]

user_item = [clothing_user_item, electronic_user_item, food_user_item]

user_bundle = [clothing_user_bundle, electronic_user_bundle, food_user_bundle]

session_bundle = [clothing_session_bundle, electronic_session_bundle, food_session_bundle]

session_item = [clothing_session_item, electronic_session_item, food_session_item]

item_names = [clothing_item_names, electronic_item_names, food_item_names]

In [9]:
def extract_json_simple_replace(response_text):
    """
    Extracts a JSON object from a string that has a "===JSON_START===" separator.

    This function isolates the JSON by finding the first '{' and last '}'
    to ensure it works correctly even with markdown fences or extra whitespace.

    Args:
        response_text (str): The full string containing the separator and JSON.

    Returns:
        dict: The parsed JSON object as a Python dictionary, or None if an error occurs.
    """
    try:
        # 1. Get the text after the separator
        json_part = response_text.split("===JSON_START===")[1]

        # 2. Find the boundaries of the JSON object
        first_brace = json_part.find('{')
        last_brace = json_part.rfind('}')

        # 3. Slice the string to get only the valid JSON
        # This will fail gracefully in the json.loads() if a brace isn't found
        json_string = json_part[first_brace : last_brace + 1]

        # 4. Parse the clean string
        parsed_json = json.loads(json_string)
        return parsed_json

    except IndexError:
        print("Error: The separator '===JSON_START===' was not found.")
        return None
    except json.JSONDecodeError:
        print("Error: Could not find or parse a valid JSON object after the separator.")
        return None


def extract_reasoning_part(response_text: str) -> str:
    """
    Extracts the reasoning text that appears before a "===JSON_START===" separator.

    Args:
        response_text: The full string from the LLM.

    Returns:
        The text before the separator, with leading/trailing whitespace removed.
        Returns the full string if the separator is not found.
    """
    # .split(...)[0] is always safe and does not need a try/except block.
    reasoning_part = response_text.split("===JSON_START===")[0]
    return reasoning_part.strip()


def extract_json_from_response(response_text):
    # Find the index of the first '{' and the last '}'
    start_index = response_text.find('{')
    end_index = response_text.rfind('}')

    if start_index != -1 and end_index != -1:
        # Slice the string to get only the content between the brackets
        json_string = response_text[start_index : end_index + 1]
        return json_string
    else:
        # Return None if a valid JSON block is not found
        return None


import numpy as np
from numba import jit # accelerate NumPy computations without changing code

@jit(nopython=True)
def matrix_cosine_line_jit(embeddings, new_embedding):
  """
  Compute the cosine similarity matrix using JIT (Numba).

  :param embeddings: NumPy array of embeddings
  :return: NumPy matrix of cosine similarities
  """

  n = embeddings.shape[0]


  norms = np.sqrt(np.sum(embeddings ** 2, axis=1))
  new_norm = np.sqrt(np.sum(new_embedding ** 2))

  cosine_similarity_line = np.zeros((n), dtype=np.float32) # Initialise the array to fix the memory required.

  for i in range(n):
      cosine_similarity_line[i] = np.dot(embeddings[i], new_embedding) / (norms[i] * new_norm)


  return cosine_similarity_line


def find_k_neighbours(new_item, embeddings, domain_items, k):

    new_embedding = model.encode(new_item, convert_to_tensor=False)

    sim_line = matrix_cosine_line_jit(embeddings, new_embedding)

    # Get indices of top 3 values
    top_k_indices = np.argsort(sim_line)[-k:][::-1]


    nearest_items = [domain_items[i] for i in top_k_indices]


    return nearest_items, top_k_indices



def input_strings(intent_list, item_list):
    l = len(intent_list)
    string_list = []
    for i in range(l):
        intent = intent_list[i]
        items = item_list[i]

        item_str = "\n".join([f"{i + 1}. {title}" for i, title in enumerate(items)])

        string_list.append(f"Intent: {intent}\nBundle Items:\n{item_str}\n")

    return string_list

def extract_bundle_score(response):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            given_score = float(json_schema['score'])
        except:
            given_score = None
    else:
        given_score = None

    return given_score

def extract_bundle_verdict(response, consideration):
    json_schema = extract_json_simple_replace(response)
    if json_schema is not None:
        try:
            if consideration == "1-2":
                given_verdict_1 = json_schema['is_poor_quality_bundle']
                given_verdict_2 = json_schema['is_acceptable_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "3":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_good_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            elif consideration == "4-5":
                given_verdict_1 = json_schema['needs_improvement_bundle']
                given_verdict_2 = json_schema['is_high_quality_bundle']
                given_verdict = [given_verdict_1.lower(), given_verdict_2.lower()]
            else:
                given_verdict = None
        except:
            given_verdict = None
    else:
        given_verdict = None

    return given_verdict


def making_product_type_list(category_indices, all_products):
    product_list = []

    # Your original loop to gather all the products
    for i in category_indices:
        product_list = product_list + all_products[i]

    # Convert to a set to remove duplicates, then convert back to a list
    unique_product_list = list(set(product_list))

    return unique_product_list


def filter_decision(bad_verdicts, bad_scores, middle_verdicts, middle_scores, good_verdicts, good_scores, score_acceptability = [3.4, 3.9]):

    l = len(bad_verdicts)

    unsure = []



    all_evaluators = []

    all_verdicts = []


    modify = []
    retain = []


    min_scores = [min(bad_scores[i], middle_scores[i], good_scores[i]) for i in range(l)]
    average_scores = [np.round((bad_scores[i] + middle_scores[i] + good_scores[i])/3, 2) for i in range(l)]



    for i in range(l):
        all_evaluators.append([bad_scores[i], middle_scores[i], good_scores[i]])
        all_verdicts.append([bad_verdicts[i], middle_verdicts[i], good_verdicts[i]])




        if bad_verdicts[i][0] == "yes" and bad_verdicts[i][1] == "no":
            bad_verd = "yes"
        elif bad_verdicts[i][0] == "no" and bad_verdicts[i][1] == "yes":
            bad_verd = "no"
        else:
            bad_verd = None
            print("bad verdict error")

        if middle_verdicts[i][0] == "yes" and middle_verdicts[i][1] == "no":
            middle_verd = "yes"
        elif middle_verdicts[i][0] == "no" and middle_verdicts[i][1] == "yes":
            middle_verd = "no"
        else:
            middle_verd = None
            print("middle verdict error")

        if good_verdicts[i][0] == "no" and good_verdicts[i][1] == "yes":
            good_verd = "yes"
        elif good_verdicts[i][0] == "yes" and good_verdicts[i][1] == "no":
            good_verd = "no"
        else:
            good_verd = None
            print("good verdict error")



        if bad_verd == "no" and middle_verd == "no" and good_verd == "yes":
            retain.append(i)
        else:
            unsure.append(i)


    # print(len(unsure))


    for i in unsure:
        bad_score = bad_scores[i]
        middle_score = middle_scores[i]
        good_score = good_scores[i]

        min_score = min_scores[i]
        avg_score = average_scores[i]


        # if (bad_score + middle_score + good_score)/3 > 3.5:
        if min_score > score_acceptability[0] and avg_score > score_acceptability[1]:
            retain.append(i)
        # elif (bad_score + middle_score + good_score)/3 < 3.5:
        else:
            modify.append(i)



    return modify, retain, all_evaluators, all_verdicts, average_scores, min_scores


def bundle_info_provider(bundle_ID, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    original_session_ID = session_bundle[k].iloc[bundle_ID]["session ID"]

    item_ids = session_item[k][session_item[k]["session ID"] == original_session_ID]["item ID"].values

    # A list to store the titles of the items that were successfully found
    session_item_titles = []


    existing = []

    for item_id in item_ids:
        item_titles = item_names[k][item_names[k]["item ID"] == item_id]["titles"]
        if not item_titles.empty and item_titles.values[0] not in existing:
            session_item_titles.append(item_titles.values[0])

            existing.append(item_titles.values[0])

    session_item_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in session_item_titles]

    bundle_list = bundle_items_list[k]

    bundle_item_ids = [metadata[k][metadata[k]['titles'] == names].index.tolist()[0] for names in bundle_list[bundle_ID]]

    bundle_item_titles =  [metadata[k].iloc[i]['titles'] for i in bundle_item_ids]

    bundle_intent = intents[k].iloc[bundle_ID]['intent']

    return bundle_intent, bundle_item_ids, bundle_item_titles, session_item_ids, session_item_titles

def bundle_token_strings(intent_list, item_list, index_list, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    l = len(intent_list)
    string_list = []

    for i in range(l):
        intent = intent_list[i]
        items = item_list[i]
        indexes = index_list[i]
        meta_list = [metadata_jsons[k][j] for j in indexes]

        token_list = []

        for m in range(len(indexes)):

            metasetter = json.loads(meta_list[m])


            parts = []

            if k in [0, 1]:  # clothing or electronic
                standard_keys = ['product_type', 'brand', 'design_focus', 'cost_tier']
                list_keys = ['key_features']
            else:  # food
                standard_keys = ['product_type', 'brand', 'flavor_profile', 'cost_tier']
                # The order from your original code is preserved
                list_keys = ['dietary_considerations', 'key_features']

            for key in standard_keys:
                value = metasetter.get(key)
                if value:  # This check skips None or empty string values
                    parts.append(str(value))

            for key in list_keys:
            # Default to an empty list `[]` if the key is missing.
                feature_list = metasetter.get(key, [])
                if isinstance(feature_list, list):
                    parts.extend(feature_list)

            if parts:
                # Join all the found parts into the final [part1][part2]... format
                search_string = "".join([f"[{part}]" for part in parts])

            token_list.append(search_string)

            # key_features = metasetter['key_features']



            # added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['design_focus'] + "][" + metasetter['cost_tier'] + "]["# + metasetter['key_features']
            # for j in range(len(key_features)):
            #     added_str = added_str + key_features[j] + "]["


            # token_list.append(added_str[0:-1])


        item_str = "\n".join([f"[ID: {indexes[j]}]. {items[j]}\n{token_list[j]}\n" for j in range(len(indexes))])

        string_list.append(f"Intent: {intent}\nBundle Items:\n\n{item_str}\n")




    return string_list


def candidate_token_strings(item_list, index_list, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    l = len(index_list)


    token_list = []

    for i in range(l):

        items = item_list[i]
        index = index_list[i]
        meta_list = metadata_jsons[k][index]





        metasetter = json.loads(meta_list)

        parts = []

        if k in [0, 1]:  # clothing or electronic
            standard_keys = ['product_type', 'brand', 'design_focus', 'cost_tier']
            list_keys = ['key_features']
        else:  # food
            standard_keys = ['product_type', 'brand', 'flavor_profile', 'cost_tier']
            # The order from your original code is preserved
            list_keys = ['dietary_considerations', 'key_features']

        for key in standard_keys:
            value = metasetter.get(key)
            if value:  # This check skips None or empty string values
                parts.append(str(value))

        for key in list_keys:
        # Default to an empty list `[]` if the key is missing.
            feature_list = metasetter.get(key, [])
            if isinstance(feature_list, list):
                parts.extend(feature_list)

        if parts:
            # Join all the found parts into the final [part1][part2]... format
            search_string = "".join([f"[{part}]" for part in parts])

        token_list.append(search_string)

        # key_features = metasetter['key_features']



        # added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['design_focus'] + "][" + metasetter['cost_tier'] + "]["# + metasetter['key_features']
        # for j in range(len(key_features)):
        #     added_str = added_str + key_features[j] + "]["


        # token_list.append(added_str[0:-1])


    item_str = "\n".join([f"[ID: {index_list[j]}]. {item_list[j]}\n{token_list[j]}\n" for j in range(l)])

    string_list = f"Candidate Items:\n\n{item_str}\n"




    return string_list


def stochastic_decision(bundle_items):
    l = len(bundle_items)
    if l == 2:
        return "add"
    elif l > 8:
        return "remove"
    else:
        chance_to_add = 0.65 + (3 - l)*0.1

        if random.random() < chance_to_add:
            return "add"
        else:
            return "remove"


In [ ]:
async def single_request(user, system=None, seed_value=None):

    if system:
        message = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    else:
        message = [{"role": "user", "content": user}]

    # Reimplemented the robust retry loop with exponential backoff
    for delay_secs in (2**x for x in range(0, 3)):
        try:
            response = await async_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=message,
                temperature=0,
                max_tokens=8000,
                seed=seed_value
            )
            return response.choices[0].message.content.strip()
        except openai.OpenAIError as e:
            randomness_collision_avoidance = random.randint(0, 1000) / 1000.0
            sleep_dur = delay_secs + randomness_collision_avoidance
            print(f"Error: {e}. Retrying in {round(sleep_dur, 2)} seconds.")
            await asyncio.sleep(sleep_dur)

    # Return None if all retries fail
    return None


async def openai_request(prompts, system=None, batch_size=128, delay=0):

    results = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        tasks = [
            single_request(d["prompts"], system=system, seed_value=42)
            for j, d in enumerate(batch)
        ]

        batch_results = await asyncio.gather(*tasks)
        results.extend(batch_results)
        print(f"✅ Sending batch {i // batch_size + 1} — sleeping for {delay}s...\n")
        await asyncio.sleep(delay)

    return results


async def separate_consideration_scores(char, initial_prompt, sample_data, json_addition, constant_metrics = "", consideration = ""):

    if consideration == "":
        return

    else:

        prompt_list = [{"prompts": data + "\n" + json_addition} for data in sample_data]

        responses = await openai_request(prompt_list, initial_prompt + constant_metrics)

        if consideration == "":
            scores = [extract_bundle_score(i) for i in responses]

            verdicts = None

        elif consideration == "1-2" or consideration == "3" or consideration == "4-5":
            verdicts = [extract_bundle_verdict(i, consideration) for i in responses]

            scores = [extract_bundle_score(i) for i in responses]


        filename = f"/content/drive/MyDrive/EGPO/filter_results/{char}.pkl"

        with open(filename, 'wb') as f:
            pickle.dump([responses, verdicts, scores], f)

        return responses, verdicts, scores




async def evaluation_module(charizards, prompts, input_strings):
    bad_responses, bad_verdicts, bad_scores = await separate_consideration_scores(char= charizards[0],
                                  initial_prompt= prompts[0],
                                  sample_data = input_strings,
                                  json_addition = json_bad,
                                  constant_metrics = adding_metrics,
                                  consideration = "1-2")


    middle_responses, middle_verdicts, middle_scores = await separate_consideration_scores(char= charizards[1],
                                  initial_prompt= prompts[1],
                                  sample_data = input_strings,
                                  json_addition = json_middle,
                                  constant_metrics = adding_metrics,
                                  consideration = "3")

    good_responses, good_verdicts, good_scores = await separate_consideration_scores(char= charizards[2],
                                  initial_prompt= prompts[2],
                                  sample_data = input_strings,
                                  json_addition = json_good,
                                  constant_metrics = adding_metrics,
                                  consideration = "4-5")


    modify, retain, all_evaluators, all_verdicts, all_scores, terribles = filter_decision(bad_verdicts, bad_scores, middle_verdicts, middle_scores, good_verdicts, good_scores)


    # summary_prompts = [summary_prompt(input_strings[i], extract_reasoning_part(bad_responses[i]), extract_reasoning_part(middle_responses[i]), extract_reasoning_part(good_responses[i]), all_scores[i]) for i in modify]

    summary_prompts = [drugs_prompt(input_strings[i], extract_reasoning_part(bad_responses[i]), extract_reasoning_part(middle_responses[i]), extract_reasoning_part(good_responses[i]), all_scores[i]) for i in modify]

    summary_prompts = [{"prompts": summary_prompt} for summary_prompt in summary_prompts]

    summary_responses = await openai_request(summary_prompts, "")

    return modify, retain, all_scores, summary_responses


In [ ]:
def drugs_prompt(bundle_str, reasoning1, reasoning2, reasoning3, average_score, min_score):
    """
    Generates a prompt for an LLM to analyze and propose a fix for a poorly-rated bundle.

    Args:
        bundle_items (list): A list of strings representing the items in the bundle.
        reasoning1 (str): The reasoning from the first evaluator.
        reasoning2 (str): The reasoning from the second evaluator.
        reasoning3 (str): The reasoning from the third evaluator.

    Returns:
        str: The fully formatted prompt string.
    """
    # Format the list of items into a clean string for the prompt
    # bundle_items_str = ", ".join(bundle_items)

    # Use a triple-quoted f-string to build the multi-line prompt

    adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


    prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages. Your task is to analyze a failing bundle and capture all nuances given by the experts.

**## CONTEXT:**
You have received a bundle that was reviewed by three different evaluators. The bundle's quality score is not high enough.


{bundle_str}
* **Average Score:** {average_score}
* **Min Score:** {min_score}

* **Evaluator 1 Reasoning:**
{reasoning1}

* **Evaluator 2 Reasoning:**
{reasoning2}

* **Evaluator 3 Reasoning:**
{reasoning3}

---

**## YOUR TASK: Problem Analysis**

You must perform the following four steps:

1.  **Synthesize the Core Problem:** Read all three evaluator reasonings and combine them into a single, concise **Problem Statement**. This statement should identify the bundle's single biggest flaw (e.g., "The bundle lacks a cohesive theme," "It's missing an essential item," or "An item is incompatible with the rest").

2.  **List the Top 3 Flaws:** Based on your analysis and the evaluator feedback, list the three most significant reasons that contribute to the core problem you identified.

3. **Evaluate Metrics:** Give verdict of low, medium, high for each metric below.
{adding_metrics}

4. **Operation Decision:** Decide whether to add or remove items from the bundle depending on the verdict of each metric.
If low Functionality Integration: Add or remove items to be able to fulfill intended purpose of bundle.
If low Similarity: Add more items that are thematic to the intent.
If low Complementarity: Remove the noisy item.
If low Diversity: Add a more niche item that is thematic to the intent.

---

**## OUTPUT FORMAT:**
After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "problem_summary": "string",
  "Functionality Integration: string, must be low/medium/high",
  "Similarity: string, must be low/medium/high",
  "Complementarity: string, must be low/medium/high",
  "Diversity: string, must be low/medium/high",
  "operation": "string, must be add/remove ONLY PICK ONE",
  "new_bundle_intent": "string,  a concise 3 or 4 word summary of the intended direction of the new, improved bundle's theme and purpose. The intent must be specific and not a broad category. Do not include phrases like 'and accessories' or 'and tools'."
}}
````
"""

    return prompt




def get_drugs(summary_response, bundle_items, bundle_intent, randomise = ""):

    reasoning_part = extract_reasoning_part(summary_response)

    json_schema = extract_json_simple_replace(summary_response)

    if json_schema is None:
        return reasoning_part, stochastic_decision(bundle_items), bundle_intent


    try:

        decision = json_schema['operation'].lower()
        intended_direction = json_schema['new_bundle_intent']

        if decision not in ["add", "remove"] or randomise == "yes":
            # print(randomise)
            another_decision = stochastic_decision(bundle_items)

            return reasoning_part, another_decision, intended_direction

        else:
            return reasoning_part, decision, intended_direction

    except:
        return reasoning_part, stochastic_decision(bundle_items), bundle_intent


def create_remove_item_prompt(bundle_str, summary, average_score, min_score):
    """
    Generates a prompt for an LLM to analyze and propose a fix for a poorly-rated bundle.

    Args:
        bundle_items (list): A list of strings representing the items in the bundle.
        reasoning1 (str): The reasoning from the first evaluator.
        reasoning2 (str): The reasoning from the second evaluator.
        reasoning3 (str): The reasoning from the third evaluator.

    Returns:
        str: The fully formatted prompt string.
    """
    # Format the list of items into a clean string for the prompt
    # bundle_items_str = ", ".join(bundle_items)

    # Use a triple-quoted f-string to build the multi-line prompt
    prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages. Your task is to analyze a failing bundle and decide if the REMOVE operation should be applied.

**## CONTEXT:**
You have received a bundle that was reviewed by three different evaluators. The bundle's quality score is not high enough.


{bundle_str}
* **Average Score:** {average_score}
* **Min Score:** {min_score}

* **Evaluator Summary:**
{summary}

---

**## YOUR TASK:**
You must perform the following two steps.

**Part 1: Summarize the Core Problem**
First, look at the evaluator reasoning summary and try to determine whether or not removing a **SINGLE** item would fix each of the main problems of the bundle.

**Part 2: Decide whether or not to apply Remove Operation**
Based on your analysis of the problem, decide whether or not to: `REMOVE` (yes/no).

---

**## OUTPUT FORMAT:**
After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "problem_summary": "string",
  "remove_item": "string, must be yes/no",
  "item_to_remove": "string, required for REMOVE, otherwise null if remove item is unnecessary",
  "item_to_remove_id": "int, required for REMOVE, otherwise null if operation is ADD",
  "new_bundle_intent": "string, a concise 3-5 word summary of the new, improved bundle's theme and purpose."
}}
```
"""

    return prompt


def bundle_item_remover(bundle_items, bundle_indices, remove_response):

    json_schema = extract_json_simple_replace(remove_response)

    if json_schema is None:
        return bundle_items, bundle_indices, None


    try:

        decision = json_schema['remove_item'].lower()

        if decision == "yes":
            item_to_remove = json_schema['item_to_remove']
            item_to_remove_id = json_schema['item_to_remove_id']
            new_bundle_intent = json_schema['new_bundle_intent']

            if item_to_remove in bundle_items:
                index_to_remove = bundle_items.index(item_to_remove)
                bundle_items.pop(index_to_remove)
                bundle_indices.pop(index_to_remove)
                return bundle_items, bundle_indices, new_bundle_intent

            elif item_to_remove_id in bundle_indices:
                index_to_remove = bundle_indices.index(item_to_remove_id)
                bundle_items.pop(index_to_remove)
                bundle_indices.pop(index_to_remove)
                return bundle_items, bundle_indices, new_bundle_intent

            else:
                return bundle_items, bundle_indices, None


        elif decision == "no":
            return bundle_items, bundle_indices, None

    except:
        return bundle_items, bundle_indices, None


    return bundle_items, bundle_indices, None


def create_expand_candidates_prompt(bundle_str, summary, product_types, domain):
    """
    Generates a prompt for an LLM to propose three distinct, single-item additions to a bundle.

    Args:
        bundle_str (str): A formatted string of the items currently in the bundle.
        summary (str): A summary of the evaluators' reasoning for adding an item.
        product_types_str (str): A formatted string of the candidate product types.

    Returns:
        str: The fully formatted prompt string.
    """

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    # Use a triple-quoted f-string to build the multi-line prompt
    product_types_str = '\n'.join([f"- {j}" for j in product_types])


    if k == 0 or k == 1:
        prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages.

    **## CONTEXT:**
    Experts have decided that the following bundle requires a new item to be added.

    {bundle_str}

    Experts have provided an analysis summary for this bundle:
    {summary}

    ---

    **## YOUR TASK:**
    Your goal is to propose **three different and independent suggestions** for a single item to add to this bundle. Each suggestion should represent a potentially different way to improve the bundle.

    1. **Propose Three Product Types:** Based on the context, choose three distinct product types from the list below that would be good single additions.
    2. **Generate Ideal Characteristics:** For each of the three product types you chose, generate a full profile of characteristics (brand, design focus, etc.) that would make it a perfect fit for the existing bundle.

    **Candidate Product Types:**
    {product_types_str}

    ---

    **## OUTPUT FORMAT:**
    After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your three suggestions. Do not include any other text after the separator.

    **JSON Schema:**
    The output must be a JSON object with a single key "suggestions" which contains a list of three objects. Each object represents one ideal product to add.
    ```json
    {{
      "suggestions": [
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }}
      ]
    }}
    ```"""

    elif k == 2:
        prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages.

    **## CONTEXT:**
    Experts have decided that the following bundle requires a new item to be added.

    {bundle_str}

    Experts have provided an analysis summary for this bundle:
    {summary}

    ---

    **## YOUR TASK:**
    Your goal is to propose **three different and independent suggestions** for a single item to add to this bundle. Each suggestion should represent a potentially different way to improve the bundle.

    1. **Propose Three Product Types:** Based on the context, choose three distinct product types from the list below that would be good single additions.
    2. **Generate Ideal Characteristics:** For each of the three product types you chose, generate a full profile of characteristics (brand, design focus, etc.) that would make it a perfect fit for the existing bundle.

    **Candidate Product Types:**
    {product_types_str}

    ---

    **## OUTPUT FORMAT:**
    After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your three suggestions. Do not include any other text after the separator.

    **JSON Schema:**
    The output must be a JSON object with a single key "suggestions" which contains a list of three objects. Each object represents one ideal product to add.
    ```json
    {{
      "suggestions": [
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }}
      ]
    }}
    ```"""

    return prompt


def extended_candidates_extractor(candidate_responses, domain, num_candidates=2):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    l = len(candidate_responses)

    candidate_jsons = [extract_json_simple_replace(candidate_responses[i]) for i in range(l)]

    # A list to hold our final formatted strings
    collected_items = []
    collected_ids = []

    for i in range(l):

        json_string = candidate_jsons[i]

        collated_items = []
        collated_ids = []

        if json_string is None:
            print(f"Warning: No valid JSON found for response index {i}. Skipping.")
            collected_items.append([])
            collected_ids.append([])
            continue  # Safely skip to the next item in the loop

        suggestions = json_string.get('suggestions', [])

        # Loop through each suggestion dictionary in the 'suggestions' list
        for metasetter in suggestions:

            parts = []

            if k in [0, 1]:  # clothing or electronic
                standard_keys = ['product_type', 'brand', 'design_focus', 'cost_tier']
                list_keys = ['key_features']
            else:  # food
                standard_keys = ['product_type', 'brand', 'flavor_profile', 'cost_tier']
                # The order from your original code is preserved
                list_keys = ['dietary_considerations', 'key_features']

            for key in standard_keys:
                value = metasetter.get(key)
                if value:  # This check skips None or empty string values
                    parts.append(str(value))

            for key in list_keys:
            # Default to an empty list `[]` if the key is missing.
                feature_list = metasetter.get(key, [])
                if isinstance(feature_list, list):
                    parts.extend(feature_list)

            if parts:
                # Join all the found parts into the final [part1][part2]... format
                search_string = "".join([f"[{part}]" for part in parts])

                nearest_items, top_k_indices = find_k_neighbours(
                    search_string,
                    super_embeddings[k],
                    domain_items[k],
                    num_candidates
                )

                collated_items = collated_items + nearest_items
                collated_ids = collated_ids + list(top_k_indices)


                # added_str = "[" + metasetter['product_type'] + "][" + metasetter['brand'] + "][" + metasetter['design_focus'] + "][" + metasetter['cost_tier'] + "]["# + metasetter['key_features']
                # for j in range(len(key_features)):
                #     added_str = added_str + key_features[j] + "]["


                # Join all the parts together into the final [part1][part2]... format

                # nearest_items, top_k_indices = find_k_neighbours(added_str[0:-1], super_embeddings[k], domain_items[k], num_candidates)


        collected_items.append(collated_items)
        collected_ids.append(collated_ids)

    return collected_items, collected_ids

def extended_candidates_mixer(bundle_items, session_items, session_ids, extended_items=None, extended_ids=None):

    if extended_items is None:
        extended_items = []
    if extended_ids is None:
        extended_ids = []

    candidate_items = copy.deepcopy(session_items)
    candidate_ids = copy.deepcopy(session_ids)

    # print(candidate_items)

    # print(candidate_ids)

    for i in range(len(extended_items)):
        if extended_items[i] not in candidate_items:
            candidate_items.append(extended_items[i])
            candidate_ids.append(extended_ids[i])

    vetted_items, vetted_ids = [], []

    for i in range(len(candidate_items)):
        if candidate_items[i] not in bundle_items:
            vetted_items.append(candidate_items[i])
            vetted_ids.append(candidate_ids[i])

    return vetted_items, np.array(vetted_ids).tolist()

def create_add_item_prompt(bundle_str, summary, additional_items):
    """
    Generates a prompt for an LLM to choose a single best item to add to a bundle
    and define the new bundle's intent.
    """

    prompt = f"""You are an Expert Bundle Designer tasked with improving an existing bundle.

**## CONTEXT:**
You are analyzing a bundle that needs one new item.

{bundle_str}

Experts have provided analysis to aid you in the addition operation:
{summary}

{additional_items}

---

**## YOUR TASK:**
You must perform the following two parts.

**Part 1: Multi-Scenario Analysis**
For **EACH** item in the 'Candidate Items to Add' list, create a separate, hypothetical "Importance Graph". This graph should show how the bundle would be ranked if that specific candidate were added. Follow this format for each candidate:

---
**Importance Graph for adding [Candidate Item Name]:**
a. [Item 1 from bundle or candidate] — [Role: Primary/Secondary/Tertiary]
   Reason: [Your explanation for this item's rank in this specific scenario]
b. [Item 2 from bundle or candidate] — [Role: Primary/Secondary/Tertiary]
   Reason: [Your explanation]
(...and so on for all original items plus the one candidate)
---

**Part 2: Final Recommendation**
After you have created an Importance Graph for every candidate, compare the outcomes. Decide which addition creates the most cohesive and valuable bundle overall. Summarize your final decision in the JSON format below.

---

**## OUTPUT FORMAT:**
First, provide your full written analysis from Part 1, showing the multiple Importance Graphs. After that is complete, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your final recommendation from Part 2. Do not include any other text after the JSON.

**JSON Schema:**
```json
{{
  "reasoning_for_choice": "string, your detailed justification that explains WHY you chose the final item after considering all scenarios.",
  "chosen_item_to_add": "string, the single best item you selected from the candidate list.",
  "chosen_item_id": "int, the ID of the chosen item.",
  "new_bundle_intent": "string, a concise 3-5 word summary of the new, improved bundle's theme and purpose."
}}
```"""

    return prompt


def bundle_item_adder(bundle_intent, bundle_items, bundle_indices, add_response, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    json_schema = extract_json_simple_replace(add_response)

    if json_schema is None:
        return bundle_intent, bundle_items, bundle_indices


    try:
        chosen_item_to_add = json_schema['chosen_item_to_add']
        chosen_item_id = json_schema['chosen_item_id']
        new_bundle_intent = json_schema['new_bundle_intent']
    except:
        return bundle_intent, bundle_items, bundle_indices


    matching_rows = metadata[k][metadata[k]['titles'] == chosen_item_to_add]

    if not matching_rows.empty:
        if chosen_item_to_add not in bundle_items:

            bundle_items.append(chosen_item_to_add)
            bundle_indices.append(matching_rows.index[0])

            return new_bundle_intent, bundle_items, bundle_indices


    try:
        new_bundle_item = metadata[k].iloc[chosen_item_id]['titles']

        if new_bundle_item not in bundle_items:
            bundle_items.append(new_bundle_item)
            bundle_indices.append(chosen_item_id)

        return new_bundle_intent, bundle_items, bundle_indices

    except:

        return new_bundle_intent, bundle_items, bundle_indices




    return bundle_intent, bundle_items, bundle_indices


In [10]:
with open("/content/LLM4BEAR/BundleRec Data/enriched_outputs_electronic.pkl", "rb") as f:
    text_electronics = pickle.load(f)

print(text_electronics[3498])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_clothing.pkl", "rb") as f:
    text_clothing = pickle.load(f)

print(text_clothing[3000])

with open(f"/content/LLM4BEAR/BundleRec Data/enriched_outputs_food.pkl", "rb") as f:
    text_food = pickle.load(f)

print(text_food[3000])



electronic_category_keys = [
    'Camera and Accessories', 'Computers and Accessories', 'Audio Equipment',
    'Tablets and Accessories', 'Storage Solutions', 'Networking Equipment',
    'Mobile Devices and Accessories', 'Travel Accessories', 'Gaming',
    'Home Entertainment Systems', 'Miscellaneous Electronics', 'Power Solutions',
    'Cables and Connectors', 'Security Systems', 'Car Technology and Accessories',
    'Photography and Camera Equipment', 'Adapters and Cables',
    'Television and Accessories', 'AV Setup', 'GPS and Navigation Accessories',
    'Mobile Device Protection', 'Streaming and Media', 'PC Building and Assembly',
    'General Electronics', 'Walkie Talkies and Communication Devices'
]

clothing_category_keys = [
    'Footwear', 'Accessories', 'Costumes and Themed Apparel',
    'Lingerie and Underwear', 'Baby and Kids Clothing', 'Activewear and Sportswear',
    'Fashion Accessories', 'Seasonal and Thematic Products', 'Electronics',
    'Children\'s Items', 'Carrying Items', 'Maintenance', 'Clothing'
]

food_category_keys = [
    'Snacks', 'Beverages', 'Cooking Ingredients',
    'Breakfast Foods', 'Sweets and Desserts', 'Health Foods',
    'Canned and Packaged Foods', 'Baby Food', 'Condiments and Sauces',
    'Fruits and Vegetables', 'Specialty Foods', 'Dried and Preserved Foods',
    'Grains and Pasta', 'Miscellaneous', 'Gift Baskets and Food Gifts',
    'Dietary Specific Items', 'Cooking Tools and Kitchen Goods',
    'Sweeteners', 'Nuts and Seeds', 'Coffee/Tea',
    'Vegetables and Beans', 'Health-Conscious Options', 'Prepared and Ready-Made Meals',
    'Ethnic and Specialty Foods', 'Culinary Specialties'
]


with open(f"/content/LLM4BEAR/BundleRec Data/specific_electronic_product_metadata.pkl", 'rb') as f:
    metadata_electronic_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_clothing_product_metadata.pkl", 'rb') as f:
    metadata_clothing_jsons = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/specific_food_product_metadata.pkl", 'rb') as f:
    metadata_food_jsons = pickle.load(f)

metadata_jsons = [metadata_clothing_jsons, metadata_electronic_jsons, metadata_food_jsons]

with open(f"/content/LLM4BEAR/BundleRec Data/general_electronic_product_list.pkl", 'rb') as f:
    general_electronic_product_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_clothing_product_list.pkl", 'rb') as f:
    general_clothing_product_list = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/general_food_product_list.pkl", 'rb') as f:
    general_food_product_list = pickle.load(f)

electronic_items = [electronic_metadata.iloc[i]['titles'] for i in range(len(text_electronics))]

clothing_items = [clothing_metadata.iloc[i]['titles'] for i in range(len(text_clothing))]

food_items = [food_metadata.iloc[i]['titles'] for i in range(len(text_food))]


domain_items = [clothing_items, electronic_items, food_items]

super_clothing_embeddings, super_food_embeddings = [], []

super_electronic_embeddings = np.load("/content/LLM4BEAR/2_Bundle Refinement/electronic_item_token_embeddings.npy")

super_clothing_embeddings = np.load("/content/LLM4BEAR/2_Bundle Refinement/clothing_item_token_embeddings.npy")

super_food_embeddings = np.load("/content/LLM4BEAR/2_Bundle Refinement/food_item_token_embeddings.npy")

super_embeddings = [super_clothing_embeddings, super_electronic_embeddings, super_food_embeddings]

with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_electronic.pkl", 'rb') as f:
    all_electronic_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_electronic.pkl", 'rb') as f:
    electronic_cat_guys = pickle.load(f)

electronic_cat_guys = [extract_json_from_response(i) for i in electronic_cat_guys]

electronic_category_indices = []

for i in electronic_cat_guys:

    dictionary = json.loads(i)

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(electronic_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    electronic_category_indices.append(indices)

electronic_product_type_list = [making_product_type_list(electronic_category_indices[i], all_electronic_products) for i in range(len(electronic_cat_guys))]

with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_clothing.pkl", 'rb') as f:
    all_clothing_products = pickle.load(f)

with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_clothing.pkl", 'rb') as f:
    clothing_cat_guys = pickle.load(f)

clothing_cat_guys = [extract_json_from_response(i) for i in clothing_cat_guys]


clothing_category_indices = []

for i in clothing_cat_guys:

    dictionary = json.loads(i)

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(clothing_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    clothing_category_indices.append(indices)

clothing_product_type_list = [making_product_type_list(clothing_category_indices[i], all_clothing_products) for i in range(len(clothing_cat_guys))]


with open(f"/content/LLM4BEAR/BundleRec Data/all_product_types_food.pkl", 'rb') as f:
    all_food_products = pickle.load(f)


with open(f"/content/LLM4BEAR/BundleRec Data/item_categories_food.pkl", 'rb') as f:
    food_cat_guys = pickle.load(f)


food_cat_guys = [extract_json_from_response(i) for i in food_cat_guys]
food_cat_guys[3255] = """{
  "Snacks": 1,
  "Beverages": 0,
  "Cooking Ingredients": 0,
  "Breakfast Foods": 0,
  "Sweets and Desserts": 0,
  "Health Foods": 0,
  "Canned and Packaged Foods": 0,
  "Baby Food": 0,
  "Condiments and Sauces": 0,
  "Fruits and Vegetables": 0,
  "Specialty Foods": 0,
  "Dried and Preserved Foods": 0,
  "Grains and Pasta": 0,
  "Miscellaneous": 0,
  "Gift Baskets and Food Gifts": 0,
  "Dietary Specific Items": 0,
  "Cooking Tools and Kitchen Goods": 0,
  "Sweeteners": 0,
  "Nuts and Seeds": 0,
  "Coffee/Tea": 0,
  "Vegetables and Beans": 0,
  "Health-Conscious Options": 0,
  "Prepared and Ready-Made Meals": 0,
  "Ethnic and Specialty Foods": 0,
  "Culinary Specialties": 0
}"""

food_category_indices = []

for i in range(len(food_cat_guys)):
    # print(i)
    dictionary = json.loads(food_cat_guys[i])

    data = []
    # print(i)
    # Loop through your keys and append the values safely
    for index, key in enumerate(food_category_keys):
        # Use .get(key, 0) to get the value, or 0 if the key is missing.
        value = dictionary.get(key, 0)
        data.append(value)

    indices = [index for index, value in enumerate(data) if value == 1]
    food_category_indices.append(indices)

food_product_type_list = [making_product_type_list(food_category_indices[i], all_food_products) for i in range(len(food_cat_guys))]

This device is a plug-and-play USB adapter that provides 5.1 channel surround sound capabilities to computers without the need for an internal sound card.
This is a pink swimsuit designed for girls aged 7 to 16, featuring a sweetheart neckline and suitable for swimming and beach activities.
A fragrant blend of star anise, cloves, Chinese cinnamon, Sichuan peppercorns, and ginger, this seasoning enhances a variety of dishes with its unique sweet and savory flavor profile.


# We perform prompt optimisation with:
## 1. Expert Annotations + Score from experts
## 2. Score-only from experts
## No_experts stands for score-only signal.
### Basically all annotated training bundles comes with an expert annotations and a score out of 5. No_experts stands for only using the score/5 to guide prompt optimisation which is misleading but I'm too lazy to change it.

# Score-Only Guided Help was used, as clothing and food bundles did not benefit much from the annotations. Thus, for practicality, score-help was used for all three domains.

# In the paper ["Harsh", "Balanced", "Lenient"] == ["Bad", "Middle", "Good"] here.

In [12]:
electronic_charizards = [
                  "bad_evaluator_electronic_importance_no_experts", "middle_evaluator_electronic_importance_no_experts", "good_evaluator_electronic_importance_no_experts",              ]

electronic_charizard_indices = [1, 1, 2]

electronic_prompts_to_test = []

for char in electronic_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/2_Bundle Refinement/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    electronic_prompts_to_test.append(top_3_prompts)


electronic_bad_prompt = electronic_prompts_to_test[0][1]
electronic_middle_prompt = electronic_prompts_to_test[1][1]
electronic_good_prompt = electronic_prompts_to_test[2][2]

electronic_prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]

clothing_charizards = [
                  "bad_evaluator_clothing_importance_no_experts", "middle_evaluator_clothing_importance_no_experts", "good_evaluator_clothing_importance_no_experts",              ]

clothing_charizard_indices = [2, 2, 0]

clothing_prompts_to_test = []

for char in clothing_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/2_Bundle Refinement/final_prompts/Refined_{char}.pkl"


    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    clothing_prompts_to_test.append(top_3_prompts)


clothing_bad_prompt = clothing_prompts_to_test[0][2]
clothing_middle_prompt = clothing_prompts_to_test[1][2]
clothing_good_prompt = clothing_prompts_to_test[2][0]

clothing_prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]


food_charizards = [
                  "bad_evaluator_food_importance_no_experts", "middle_evaluator_food_importance_no_experts", "good_evaluator_food_importance_no_experts"              ]

food_charizard_indices = [0, 2, 0]

food_prompts_to_test = []

for char in food_charizards:

    # filename = f"/content/drive/MyDrive/EGPO/final_prompts/Refined_{char}.pkl"
    filename = f"/content/LLM4BEAR/2_Bundle Refinement/final_prompts/Refined_{char}.pkl"

    with open(filename, 'rb') as f:
        top_3_prompts, _ = pickle.load(f)

    food_prompts_to_test.append(top_3_prompts)


food_bad_prompt = food_prompts_to_test[0][0]
food_middle_prompt = food_prompts_to_test[1][2]
food_good_prompt = food_prompts_to_test[2][0]

food_prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]



json_bad = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "is_poor_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_acceptable_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


json_middle = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_good_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"

json_good = "After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions from Part 1. Do not include any other text after the separator.\n"\
            "**JSON Schema:**\n"\
            "```json\n"\
            "{{\n"\
            "needs_improvement_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "is_high_quality_bundle: str, yes/no response, do not provide anything other than yes or no.\n"\
            "score: float, bundle quality out of 5.\n"\
            "}}"


adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


In [ ]:

async def drugs_workflow(domain, char, prompts, starting_bundle_intents, starting_bundle_item_ids, starting_bundle_item_titles, session_items_ids = None, session_item_titles = None, starting_product_list=None, randomise = "", num_iterations=1, score_acceptability = [3.9, 4]):


    flags = [True for _ in range(len(starting_bundle_intents))]
    working_indices = [index for index, flag in enumerate(flags) if flag]



    bundle_intents = copy.deepcopy(starting_bundle_intents)
    bundle_items = copy.deepcopy(starting_bundle_item_titles)
    bundle_indices = copy.deepcopy(starting_bundle_item_ids)

    if starting_product_list is not None:
        product_types = copy.deepcopy(starting_product_list)
    else:
        product_types = None


    bundle_input_strings = input_strings(bundle_intents, bundle_items)

    modify, retain, all_scores, summary_responses, min_scores = await evaluation_module(charizards=[f"starting_workflow_bad_{char}", f"starting_workflow_middle_{char}", f"starting_workflow_good_{char}"],
                                                   prompts=prompts,
                                                   input_strings=bundle_input_strings, score_acceptability = score_acceptability)


    if summary_responses:
        print(summary_responses[0])


    for i in retain:
        flags[working_indices[i]] = False

    historical_intents = [copy.deepcopy(bundle_intents)]
    historical_bundle_items = [copy.deepcopy(bundle_items)]
    historical_bundle_indices = [copy.deepcopy(bundle_indices)]
    historical_bundle_scores = [copy.deepcopy(all_scores)]
    historical_min_scores = [copy.deepcopy(min_scores)]
    historical_flags = [copy.deepcopy(flags)]


    working_indices = [index for index, flag in enumerate(flags) if flag]

    for num_iter in range(num_iterations):
        print(f"\n===========================\n\nIteration {num_iter + 1}\n\n===========================\n")

        remove_indices = []
        remove_summary = []
        add_indices = []
        add_summary = []

        print(len(summary_responses), len(modify), len(working_indices), "\n\n", summary_responses[0])
        # print(summary_responses[1])
        # print(summary_responses[2])
        # print(summary_responses[3])



        for i in range(len(working_indices)):
            summary, operation, intent = get_drugs(summary_responses[i], bundle_items[working_indices[i]], bundle_intents[working_indices[i]], randomise)
            if operation == "remove":
                remove_indices.append(working_indices[i])
                remove_summary.append(summary)
                bundle_intents[working_indices[i]] = intent
            elif operation == "add":
                add_indices.append(working_indices[i])
                add_summary.append(summary)
                bundle_intents[working_indices[i]] = intent


        remove_scores = [all_scores[i] for i in remove_indices]
        remove_min_scores = [min_scores[i] for i in remove_indices]
        add_scores = [all_scores[i] for i in add_indices]


        remove_intents, remove_items, remove_ids = [bundle_intents[i] for i in remove_indices], [bundle_items[i] for i in remove_indices], [bundle_indices[i] for i in remove_indices]
        add_intents, add_items, add_ids = [bundle_intents[i] for i in add_indices], [bundle_items[i] for i in add_indices], [bundle_indices[i] for i in add_indices]

        remove_strings = bundle_token_strings(remove_intents, remove_items, remove_ids, domain)
        add_strings = bundle_token_strings(add_intents, add_items, add_ids, domain)


        remove_prompts = [create_remove_item_prompt(remove_strings[i], remove_summary[i], remove_scores[i], remove_min_scores[i]) for i in range(len(remove_indices))]

        if remove_prompts:

            print(remove_prompts[0])

        remove_prompts = [{"prompts": remove_prompt} for remove_prompt in remove_prompts]

        remove_responses = await openai_request(remove_prompts, "")

        if remove_responses:
            print(remove_responses[0])

        for i in range(len(remove_indices)):
            post_remove_items, post_remove_ids, post_remove_intents = bundle_item_remover(remove_items[i], remove_ids[i], remove_responses[i])
            bundle_items[remove_indices[i]] = post_remove_items
            bundle_indices[remove_indices[i]] = post_remove_ids
            # if post_remove_intents is not None:
            #     bundle_intents[remove_indices[i]] = post_remove_intents

        if product_types is not None:

            expand_candidates_prompts = [create_expand_candidates_prompt(add_strings[i], add_summary[i], product_types=product_types[add_indices[i]], domain=domain) for i in range(len(add_indices))]

            if expand_candidates_prompts:

                print(expand_candidates_prompts[0])

            expand_candidates_prompts = [{"prompts": expand_candidates_prompt} for expand_candidates_prompt in expand_candidates_prompts]

            expand_candidates_responses = await openai_request(expand_candidates_prompts, "")

            expanded_items, expanded_ids = extended_candidates_extractor(expand_candidates_responses, domain, num_candidates=2)

            candidate_strings = []

            for i in range(len(add_indices)):

                if session_items_ids is not None:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]], expanded_items[i], expanded_ids[i])

                else:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], expanded_items[i], expanded_ids[i])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            if add_prompts:

                print(add_prompts[0])

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")


            if add_responses:

                print(add_responses[0])

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids

        else:
            candidate_strings = []

            for i in range(len(add_indices)):

                candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids




        working_indices = [index for index, flag in enumerate(flags) if flag]


        modified_intents = [bundle_intents[i] for i in working_indices]
        modified_items = [bundle_items[i] for i in working_indices]
        modified_ids = [bundle_indices[i] for i in working_indices]

        modified_input_strings = input_strings(modified_intents, modified_items)

        modify, retain, adjusted_scores, summary_responses, adjusted_min_scores = await evaluation_module(charizards=[f"iterative_workflow_bad_{char}_{num_iter + 1}", f"iterative_workflow_middle_{char}_{num_iter + 1}", f"iterative_workflow_good_{char}_{num_iter + 1}"],
                                                      prompts=prompts,
                                                      input_strings=modified_input_strings, score_acceptability = score_acceptability)

        for i in range(len(adjusted_scores)):
            all_scores[working_indices[i]] = adjusted_scores[i]
            min_scores[working_indices[i]] = adjusted_min_scores[i]

        for i in retain:
            flags[working_indices[i]] = False

        working_indices = [index for index, flag in enumerate(flags) if flag]

        for i in range(len(bundle_items)):
            if len(bundle_items[i]) < 2:
                flags[i] = True


        historical_intents.append(copy.deepcopy(bundle_intents))
        historical_bundle_items.append(copy.deepcopy(bundle_items))
        historical_bundle_indices.append(copy.deepcopy(bundle_indices))
        historical_bundle_scores.append(copy.deepcopy(all_scores))
        historical_min_scores.append(copy.deepcopy(min_scores))
        historical_flags.append(copy.deepcopy(flags))



        with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}_{num_iter}.pkl", "wb") as f:
            pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}.pkl", "wb") as f:
        pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    return historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags

In [ ]:

def create_expand_candidates_prompt_no_product_info(bundle_str, summary, domain):
    """
    Generates a prompt for an LLM to propose three distinct, single-item additions to a bundle.

    Args:
        bundle_str (str): A formatted string of the items currently in the bundle.
        summary (str): A summary of the evaluators' reasoning for adding an item.
        product_types_str (str): A formatted string of the candidate product types.

    Returns:
        str: The fully formatted prompt string.
    """

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2


    if k == 0 or k == 1:
        prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages.

    **## CONTEXT:**
    Experts have decided that the following bundle requires a new item to be added.

    {bundle_str}

    Experts have provided an analysis summary for this bundle:
    {summary}

    ---

    **## YOUR TASK:**
    Your goal is to propose **three different and independent suggestions** for a single item to add to this bundle. Each suggestion should represent a potentially different way to improve the bundle.

    1. **Propose Three Product Types:** Based on the context, choose three distinct product types from the list below that would be good single additions.
    2. **Generate Ideal Characteristics:** For each of the three product types you chose, generate a full profile of characteristics (brand, design focus, etc.) that would make it a perfect fit for the existing bundle.

    ---

    **## OUTPUT FORMAT:**
    After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your three suggestions. Do not include any other text after the separator.

    **JSON Schema:**
    The output must be a JSON object with a single key "suggestions" which contains a list of three objects. Each object represents one ideal product to add.
    ```json
    {{
      "suggestions": [
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "design_focus": "string",
          "target_user": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }}
      ]
    }}
    ```"""

    elif k == 2:
        prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages.

    **## CONTEXT:**
    Experts have decided that the following bundle requires a new item to be added.

    {bundle_str}

    Experts have provided an analysis summary for this bundle:
    {summary}

    ---

    **## YOUR TASK:**
    Your goal is to propose **three different and independent suggestions** for a single item to add to this bundle. Each suggestion should represent a potentially different way to improve the bundle.

    1. **Propose Three Product Types:** Based on the context, choose three distinct product types from the list below that would be good single additions.
    2. **Generate Ideal Characteristics:** For each of the three product types you chose, generate a full profile of characteristics (brand, design focus, etc.) that would make it a perfect fit for the existing bundle.

    ---

    **## OUTPUT FORMAT:**
    After your analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your three suggestions. Do not include any other text after the separator.

    **JSON Schema:**
    The output must be a JSON object with a single key "suggestions" which contains a list of three objects. Each object represents one ideal product to add.
    ```json
    {{
      "suggestions": [
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }},
        {{
          "product_type": "string",
          "brand": "string",
          "dietary_considerations": ["string"],
          "flavor_profile": "string",
          "cost_tier": "string",
          "key_features": ["string"]
        }}
      ]
    }}
    ```"""

    return prompt

In [ ]:
def bundle_strings_no_tokens(intent_list, item_list, index_list, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    l = len(intent_list)
    string_list = []

    for i in range(l):
        intent = intent_list[i]
        items = item_list[i]
        indexes = index_list[i]



        item_str = "\n".join([f"[ID: {indexes[j]}]. {items[j]}\n" for j in range(len(indexes))])

        string_list.append(f"Intent: {intent}\nBundle Items:\n\n{item_str}\n")




    return string_list


def candidate_strings_no_tokens(item_list, index_list, domain):

    k = 0
    if domain == "clothing":
        k += 0
    elif domain == "electronic":
        k += 1
    elif domain == "food":
        k += 2

    l = len(index_list)


    item_str = "\n".join([f"[ID: {index_list[j]}]. {item_list[j]}\n" for j in range(l)])

    string_list = f"Candidate Items:\n\n{item_str}\n"




    return string_list

In [ ]:

async def no_enrichment_workflow(domain, char, prompts, starting_bundle_intents, starting_bundle_item_ids, starting_bundle_item_titles, session_items_ids = None, session_item_titles = None, starting_product_list=None, randomise = "", num_iterations=1, score_acceptability = [3.9, 4]):


    flags = [True for _ in range(len(starting_bundle_intents))]
    working_indices = [index for index, flag in enumerate(flags) if flag]



    bundle_intents = copy.deepcopy(starting_bundle_intents)
    bundle_items = copy.deepcopy(starting_bundle_item_titles)
    bundle_indices = copy.deepcopy(starting_bundle_item_ids)

    if starting_product_list is not None:
        product_types = copy.deepcopy(starting_product_list)
    else:
        product_types = None


    bundle_input_strings = input_strings(bundle_intents, bundle_items)

    modify, retain, all_scores, summary_responses, min_scores = await evaluation_module(charizards=[f"starting_workflow_bad_{char}", f"starting_workflow_middle_{char}", f"starting_workflow_good_{char}"],
                                                   prompts=prompts,
                                                   input_strings=bundle_input_strings, score_acceptability = score_acceptability)


    if summary_responses:
        print(summary_responses[0])


    for i in retain:
        flags[working_indices[i]] = False

    historical_intents = [copy.deepcopy(bundle_intents)]
    historical_bundle_items = [copy.deepcopy(bundle_items)]
    historical_bundle_indices = [copy.deepcopy(bundle_indices)]
    historical_bundle_scores = [copy.deepcopy(all_scores)]
    historical_min_scores = [copy.deepcopy(min_scores)]
    historical_flags = [copy.deepcopy(flags)]


    working_indices = [index for index, flag in enumerate(flags) if flag]

    for num_iter in range(num_iterations):
        print(f"\n===========================\n\nIteration {num_iter + 1}\n\n===========================\n")

        remove_indices = []
        remove_summary = []
        add_indices = []
        add_summary = []

        print(len(summary_responses), len(modify), "\n\n", summary_responses[0])
        # print(summary_responses[1])
        # print(summary_responses[2])
        # print(summary_responses[3])



        for i in range(len(working_indices)):
            summary, operation, intent = get_drugs(summary_responses[i], bundle_items[working_indices[i]], bundle_intents[working_indices[i]], randomise)
            if operation == "remove":
                remove_indices.append(working_indices[i])
                remove_summary.append(summary)
                bundle_intents[working_indices[i]] = intent
            elif operation == "add":
                add_indices.append(working_indices[i])
                add_summary.append(summary)
                bundle_intents[working_indices[i]] = intent


        remove_scores = [all_scores[i] for i in remove_indices]
        remove_min_scores = [min_scores[i] for i in remove_indices]
        add_scores = [all_scores[i] for i in add_indices]


        remove_intents, remove_items, remove_ids = [bundle_intents[i] for i in remove_indices], [bundle_items[i] for i in remove_indices], [bundle_indices[i] for i in remove_indices]
        add_intents, add_items, add_ids = [bundle_intents[i] for i in add_indices], [bundle_items[i] for i in add_indices], [bundle_indices[i] for i in add_indices]

        remove_strings = bundle_strings_no_tokens(remove_intents, remove_items, remove_ids, domain)
        add_strings = bundle_strings_no_tokens(add_intents, add_items, add_ids, domain)


        remove_prompts = [create_remove_item_prompt(remove_strings[i], remove_summary[i], remove_scores[i], remove_min_scores[i]) for i in range(len(remove_indices))]

        if remove_prompts:

            print(remove_prompts[0])

        remove_prompts = [{"prompts": remove_prompt} for remove_prompt in remove_prompts]

        remove_responses = await openai_request(remove_prompts, "")

        if remove_responses:
            print(remove_responses[0])

        for i in range(len(remove_indices)):
            post_remove_items, post_remove_ids, post_remove_intents = bundle_item_remover(remove_items[i], remove_ids[i], remove_responses[i])
            bundle_items[remove_indices[i]] = post_remove_items
            bundle_indices[remove_indices[i]] = post_remove_ids
            # if post_remove_intents is not None:
            #     bundle_intents[remove_indices[i]] = post_remove_intents



        expand_candidates_prompts = [create_expand_candidates_prompt_no_product_info(add_strings[i], add_summary[i], domain=domain) for i in range(len(add_indices))]

        if expand_candidates_prompts:

            print(expand_candidates_prompts[0])

        expand_candidates_prompts = [{"prompts": expand_candidates_prompt} for expand_candidates_prompt in expand_candidates_prompts]

        expand_candidates_responses = await openai_request(expand_candidates_prompts, "")

        expanded_items, expanded_ids = extended_candidates_extractor(expand_candidates_responses, domain, num_candidates=2)

        candidate_strings = []

        for i in range(len(add_indices)):

            if session_items_ids is not None:

                candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]], expanded_items[i], expanded_ids[i])

            else:

                candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], expanded_items[i], expanded_ids[i])

            candidate_strings.append(candidate_strings_no_tokens(candidate_items, candidate_ids, domain))

        add_prompts = [create_add_item_prompt(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

        if add_prompts:

            print(add_prompts[0])

        add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

        add_responses = await openai_request(add_prompts, "")


        if add_responses:

            print(add_responses[0])

        for i in range(len(add_indices)):
            post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
            # bundle_intents[add_indices[i]] = post_add_intent
            bundle_items[add_indices[i]] = post_add_items
            bundle_indices[add_indices[i]] = post_add_ids


        working_indices = [index for index, flag in enumerate(flags) if flag]


        modified_intents = [bundle_intents[i] for i in working_indices]
        modified_items = [bundle_items[i] for i in working_indices]
        modified_ids = [bundle_indices[i] for i in working_indices]

        modified_input_strings = input_strings(modified_intents, modified_items)

        modify, retain, adjusted_scores, summary_responses, adjusted_min_scores = await evaluation_module(charizards=[f"iterative_workflow_bad_{char}_{num_iter + 1}", f"iterative_workflow_middle_{char}_{num_iter + 1}", f"iterative_workflow_good_{char}_{num_iter + 1}"],
                                                      prompts=prompts,
                                                      input_strings=modified_input_strings, score_acceptability = score_acceptability)

        for i in range(len(adjusted_scores)):
            all_scores[working_indices[i]] = adjusted_scores[i]
            min_scores[working_indices[i]] = adjusted_min_scores[i]

        for i in retain:
            flags[working_indices[i]] = False

        working_indices = [index for index, flag in enumerate(flags) if flag]

        for i in range(len(bundle_items)):
            if len(bundle_items[i]) < 2:
                flags[i] = True

        historical_intents.append(copy.deepcopy(bundle_intents))
        historical_bundle_items.append(copy.deepcopy(bundle_items))
        historical_bundle_indices.append(copy.deepcopy(bundle_indices))
        historical_bundle_scores.append(copy.deepcopy(all_scores))
        historical_min_scores.append(copy.deepcopy(min_scores))
        historical_flags.append(copy.deepcopy(flags))


        with open(f"/content/drive/My Drive/Bundle_Rec_datasets/historical_bundle_changes_{domain}_{char}_{num_iter}.pkl", "wb") as f:
            pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    with open(f"/content/drive/My Drive/Bundle_Rec_datasets/historical_bundle_changes_{domain}_{char}.pkl", "wb") as f:
        pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    return historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags

In [ ]:
def no_drugs_prompt(bundle_str):
    """
    Generates a prompt for an LLM to analyze and propose a fix for a poorly-rated bundle.

    Args:
        bundle_items (list): A list of strings representing the items in the bundle.
        reasoning1 (str): The reasoning from the first evaluator.
        reasoning2 (str): The reasoning from the second evaluator.
        reasoning3 (str): The reasoning from the third evaluator.

    Returns:
        str: The fully formatted prompt string.
    """
    # Format the list of items into a clean string for the prompt
    # bundle_items_str = ", ".join(bundle_items)

    # Use a triple-quoted f-string to build the multi-line prompt

    adding_metrics = "\nFunctionality Integration: Describe how a user would utilize this collection of items to achieve their primary goal. Considering the entire workflow, is this a complete and logical set of items for the task, or is there an irrelevant or missing item? \n"\
              "Similarity: What is the common theme or category that connects these items?\n"\
              "Complementarity: Are these items more valuable together than they would be if sold separately? Does the presence of one item create a clear reason to buy the other(s)?\n"\
              "Diversity: Does the variety of items in this bundle cater to a broad set of related needs for a single user, or does the mix of items seem unfocused and random?\n"


    prompt = f"""You are an Expert Bundle Designer with years of experience creating successful and coherent product packages. Your task is to analyze a failing bundle and capture all nuances.

**## CONTEXT:**
You have received a bundle that was reviewed by three different experts. The bundle's quality score is not high enough.

---

**## YOUR TASK: Problem Analysis**

You must perform the following three steps:

1.  **List the Top 3 Flaws:** Based on your analysis, list the three most significant reasons that contribute to the core problem of the bundle.

2. **Evaluate Metrics:** Give verdict of low, medium, high for each metric below.
{adding_metrics}

3. **Operation Decision:** Decide whether to add or remove items from the bundle depending on the verdict of each metric.
If low Functionality Integration: Add more items to be able to fulfill intended purpose of bundle.
If low Similarity: Add more items that are thematic to the intent.
If low Complementarity: Remove the noisy item.
If low Diversity: Add a more niche item that is thematic to the intent.

---

**## OUTPUT FORMAT:**
After you have completed your full written analysis, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object that summarizes your conclusions. Do not include any other text after the separator.

**JSON Schema:**
```json
{{
  "problem_summary": "string",
  "Functionality Integration: string, must be low/medium/high",
  "Similarity: string, must be low/medium/high",
  "Complementarity: string, must be low/medium/high",
  "Diversity: string, must be low/medium/high",
  "operation": "string, must be add/remove ONLY PICK ONE",
  "new_bundle_intent": "string,  a concise 3 or 4 word summary of the intended direction of the new, improved bundle's theme and purpose. The intent must be specific and not a broad category. Do not include phrases like 'and accessories' or 'and tools'."
}}
````
"""

    return prompt



In [ ]:

async def no_help_evaluation_module(charizards, prompts, input_strings, score_acceptability):
    bad_responses, bad_verdicts, bad_scores = await separate_consideration_scores(char= charizards[0],
                                  initial_prompt= prompts[0],
                                  sample_data = input_strings,
                                  json_addition = json_bad,
                                  constant_metrics = adding_metrics,
                                  consideration = "1-2")


    middle_responses, middle_verdicts, middle_scores = await separate_consideration_scores(char= charizards[1],
                                  initial_prompt= prompts[1],
                                  sample_data = input_strings,
                                  json_addition = json_middle,
                                  constant_metrics = adding_metrics,
                                  consideration = "3")

    good_responses, good_verdicts, good_scores = await separate_consideration_scores(char= charizards[2],
                                  initial_prompt= prompts[2],
                                  sample_data = input_strings,
                                  json_addition = json_good,
                                  constant_metrics = adding_metrics,
                                  consideration = "4-5")


    modify, retain, all_evaluators, all_verdicts, all_scores, min_scores = filter_decision(bad_verdicts, bad_scores, middle_verdicts, middle_scores, good_verdicts, good_scores, score_acceptability = score_acceptability)



    summary_prompts = [no_drugs_prompt(input_strings[i]) for i in modify]

    summary_prompts = [{"prompts": summary_prompt} for summary_prompt in summary_prompts]

    summary_responses = await openai_request(summary_prompts, "")

    return modify, retain, all_scores, summary_responses, min_scores


In [ ]:

async def no_help_workflow(domain, char, prompts, starting_bundle_intents, starting_bundle_item_ids, starting_bundle_item_titles, session_items_ids = None, session_item_titles = None, starting_product_list=None, randomise = "", num_iterations=1, score_acceptability = [3.9, 4]):


    flags = [True for _ in range(len(starting_bundle_intents))]
    working_indices = [index for index, flag in enumerate(flags) if flag]



    bundle_intents = copy.deepcopy(starting_bundle_intents)
    bundle_items = copy.deepcopy(starting_bundle_item_titles)
    bundle_indices = copy.deepcopy(starting_bundle_item_ids)

    if starting_product_list is not None:
        product_types = copy.deepcopy(starting_product_list)
    else:
        product_types = None


    bundle_input_strings = input_strings(bundle_intents, bundle_items)

    modify, retain, all_scores, summary_responses, min_scores = await no_help_evaluation_module(charizards=[f"starting_workflow_bad_{char}", f"starting_workflow_middle_{char}", f"starting_workflow_good_{char}"],
                                                   prompts=prompts,
                                                   input_strings=bundle_input_strings, score_acceptability = score_acceptability)


    if summary_responses:
        print(summary_responses[0])


    for i in retain:
        flags[working_indices[i]] = False

    historical_intents = [copy.deepcopy(bundle_intents)]
    historical_bundle_items = [copy.deepcopy(bundle_items)]
    historical_bundle_indices = [copy.deepcopy(bundle_indices)]
    historical_bundle_scores = [copy.deepcopy(all_scores)]
    historical_min_scores = [copy.deepcopy(min_scores)]
    historical_flags = [copy.deepcopy(flags)]


    working_indices = [index for index, flag in enumerate(flags) if flag]

    for num_iter in range(num_iterations):
        print(f"\n===========================\n\nIteration {num_iter + 1}\n\n===========================\n")

        remove_indices = []
        remove_summary = []
        add_indices = []
        add_summary = []

        print(len(summary_responses), len(modify), "\n\n", summary_responses[0])
        # print(summary_responses[1])
        # print(summary_responses[2])
        # print(summary_responses[3])



        for i in range(len(working_indices)):
            summary, operation, intent = get_drugs(summary_responses[i], bundle_items[working_indices[i]], bundle_intents[working_indices[i]], randomise)
            if operation == "remove":
                remove_indices.append(working_indices[i])
                remove_summary.append(summary)
                bundle_intents[working_indices[i]] = intent
            elif operation == "add":
                add_indices.append(working_indices[i])
                add_summary.append(summary)
                bundle_intents[working_indices[i]] = intent


        remove_scores = [all_scores[i] for i in remove_indices]
        remove_min_scores = [min_scores[i] for i in remove_indices]
        add_scores = [all_scores[i] for i in add_indices]


        remove_intents, remove_items, remove_ids = [bundle_intents[i] for i in remove_indices], [bundle_items[i] for i in remove_indices], [bundle_indices[i] for i in remove_indices]
        add_intents, add_items, add_ids = [bundle_intents[i] for i in add_indices], [bundle_items[i] for i in add_indices], [bundle_indices[i] for i in add_indices]

        remove_strings = bundle_token_strings(remove_intents, remove_items, remove_ids, domain)
        add_strings = bundle_token_strings(add_intents, add_items, add_ids, domain)


        remove_prompts = [create_remove_item_prompt(remove_strings[i], remove_summary[i], remove_scores[i], remove_min_scores[i]) for i in range(len(remove_indices))]

        if remove_prompts:

            print(remove_prompts[0])

        remove_prompts = [{"prompts": remove_prompt} for remove_prompt in remove_prompts]

        remove_responses = await openai_request(remove_prompts, "")

        if remove_responses:
            print(remove_responses[0])

        for i in range(len(remove_indices)):
            post_remove_items, post_remove_ids, post_remove_intents = bundle_item_remover(remove_items[i], remove_ids[i], remove_responses[i])
            bundle_items[remove_indices[i]] = post_remove_items
            bundle_indices[remove_indices[i]] = post_remove_ids
            # if post_remove_intents is not None:
            #     bundle_intents[remove_indices[i]] = post_remove_intents

        if product_types is not None:

            expand_candidates_prompts = [create_expand_candidates_prompt(add_strings[i], add_summary[i], product_types=product_types[add_indices[i]], domain=domain) for i in range(len(add_indices))]

            if expand_candidates_prompts:

                print(expand_candidates_prompts[0])

            expand_candidates_prompts = [{"prompts": expand_candidates_prompt} for expand_candidates_prompt in expand_candidates_prompts]

            expand_candidates_responses = await openai_request(expand_candidates_prompts, "")

            expanded_items, expanded_ids = extended_candidates_extractor(expand_candidates_responses, domain, num_candidates=2)

            candidate_strings = []

            for i in range(len(add_indices)):

                if session_items_ids is not None:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]], expanded_items[i], expanded_ids[i])

                else:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], expanded_items[i], expanded_ids[i])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            if add_prompts:

                print(add_prompts[0])

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")


            if add_responses:

                print(add_responses[0])

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids

        else:
            candidate_strings = []

            for i in range(len(add_indices)):

                candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids


        working_indices = [index for index, flag in enumerate(flags) if flag]


        modified_intents = [bundle_intents[i] for i in working_indices]
        modified_items = [bundle_items[i] for i in working_indices]
        modified_ids = [bundle_indices[i] for i in working_indices]

        modified_input_strings = input_strings(modified_intents, modified_items)

        modify, retain, adjusted_scores, summary_responses, adjusted_min_scores = await no_help_evaluation_module(charizards=[f"iterative_workflow_bad_{char}_{num_iter + 1}", f"iterative_workflow_middle_{char}_{num_iter + 1}", f"iterative_workflow_good_{char}_{num_iter + 1}"],
                                                      prompts=prompts,
                                                      input_strings=modified_input_strings, score_acceptability = score_acceptability)

        for i in range(len(adjusted_scores)):
            all_scores[working_indices[i]] = adjusted_scores[i]
            min_scores[working_indices[i]] = adjusted_min_scores[i]

        for i in retain:
            flags[working_indices[i]] = False

        working_indices = [index for index, flag in enumerate(flags) if flag]

        for i in range(len(bundle_items)):
            if len(bundle_items[i]) < 2:
                flags[i] = True

        historical_intents.append(copy.deepcopy(bundle_intents))
        historical_bundle_items.append(copy.deepcopy(bundle_items))
        historical_bundle_indices.append(copy.deepcopy(bundle_indices))
        historical_bundle_scores.append(copy.deepcopy(all_scores))
        historical_min_scores.append(copy.deepcopy(min_scores))
        historical_flags.append(copy.deepcopy(flags))


        with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}_{num_iter}.pkl", "wb") as f:
            pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}.pkl", "wb") as f:
        pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    return historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags

In [ ]:

def create_add_item_prompt_no_graphs(bundle_str, summary, additional_items):
    """
    Generates a prompt for an LLM to choose a single best item to add to a bundle
    and define the new bundle's intent.
    """

    prompt = f"""You are an Expert Bundle Designer tasked with improving an existing bundle.

**## CONTEXT:**
You are analyzing a bundle that needs one new item.

{bundle_str}

Experts have provided analysis to aid you in the addition operation:
{summary}

{additional_items}

---

**## YOUR TASK:**
You must perform the following two parts.

**Part 1: Multi-Scenario Analysis**
For **EACH** item in the 'Candidate Items to Add', provide a thorough analysis of how the addition of each item would change the overall value of the bundle.


**Part 2: Final Recommendation**
After you have created a thorough analysis for every candidate, compare the outcomes. Decide which addition creates the most cohesive and valuable bundle overall. Summarize your final decision in the JSON format below.

---

**## OUTPUT FORMAT:**
First, provide your full written analysis from Part 1, showing the multiple analyses. After that is complete, print the separator string `===JSON_START===` on a new line. Then, on the next line, provide a single JSON object with your final recommendation from Part 2. Do not include any other text after the JSON.

**JSON Schema:**
```json
{{
  "reasoning_for_choice": "string, your detailed justification that explains WHY you chose the final item after considering all scenarios.",
  "chosen_item_to_add": "string, the single best item you selected from the candidate list.",
  "chosen_item_id": "int, the ID of the chosen item.",
  "new_bundle_intent": "string, a concise 3-5 word summary of the new, improved bundle's theme and purpose."
}}
```"""

    return prompt


In [ ]:

async def no_graphs_workflow(domain, char, prompts, starting_bundle_intents, starting_bundle_item_ids, starting_bundle_item_titles, session_items_ids = None, session_item_titles = None, starting_product_list=None, randomise = "", num_iterations=1, score_acceptability = [3.9, 4]):


    flags = [True for _ in range(len(starting_bundle_intents))]
    working_indices = [index for index, flag in enumerate(flags) if flag]



    bundle_intents = copy.deepcopy(starting_bundle_intents)
    bundle_items = copy.deepcopy(starting_bundle_item_titles)
    bundle_indices = copy.deepcopy(starting_bundle_item_ids)

    if starting_product_list is not None:
        product_types = copy.deepcopy(starting_product_list)
    else:
        product_types = None


    bundle_input_strings = input_strings(bundle_intents, bundle_items)

    modify, retain, all_scores, summary_responses, min_scores = await evaluation_module(charizards=[f"starting_workflow_bad_{char}", f"starting_workflow_middle_{char}", f"starting_workflow_good_{char}"],
                                                   prompts=prompts,
                                                   input_strings=bundle_input_strings, score_acceptability = score_acceptability)


    if summary_responses:
        print(summary_responses[0])


    for i in retain:
        flags[working_indices[i]] = False

    historical_intents = [copy.deepcopy(bundle_intents)]
    historical_bundle_items = [copy.deepcopy(bundle_items)]
    historical_bundle_indices = [copy.deepcopy(bundle_indices)]
    historical_bundle_scores = [copy.deepcopy(all_scores)]
    historical_min_scores = [copy.deepcopy(min_scores)]
    historical_flags = [copy.deepcopy(flags)]


    working_indices = [index for index, flag in enumerate(flags) if flag]

    for num_iter in range(num_iterations):
        print(f"\n===========================\n\nIteration {num_iter + 1}\n\n===========================\n")

        remove_indices = []
        remove_summary = []
        add_indices = []
        add_summary = []

        print(len(summary_responses), len(modify), "\n\n", summary_responses[0])
        # print(summary_responses[1])
        # print(summary_responses[2])
        # print(summary_responses[3])



        for i in range(len(working_indices)):
            summary, operation, intent = get_drugs(summary_responses[i], bundle_items[working_indices[i]], bundle_intents[working_indices[i]], randomise)
            if operation == "remove":
                remove_indices.append(working_indices[i])
                remove_summary.append(summary)
                bundle_intents[working_indices[i]] = intent
            elif operation == "add":
                add_indices.append(working_indices[i])
                add_summary.append(summary)
                bundle_intents[working_indices[i]] = intent


        remove_scores = [all_scores[i] for i in remove_indices]
        remove_min_scores = [min_scores[i] for i in remove_indices]
        add_scores = [all_scores[i] for i in add_indices]


        remove_intents, remove_items, remove_ids = [bundle_intents[i] for i in remove_indices], [bundle_items[i] for i in remove_indices], [bundle_indices[i] for i in remove_indices]
        add_intents, add_items, add_ids = [bundle_intents[i] for i in add_indices], [bundle_items[i] for i in add_indices], [bundle_indices[i] for i in add_indices]

        remove_strings = bundle_token_strings(remove_intents, remove_items, remove_ids, domain)
        add_strings = bundle_token_strings(add_intents, add_items, add_ids, domain)


        remove_prompts = [create_remove_item_prompt(remove_strings[i], remove_summary[i], remove_scores[i], remove_min_scores[i]) for i in range(len(remove_indices))]

        if remove_prompts:

            print(remove_prompts[0])

        remove_prompts = [{"prompts": remove_prompt} for remove_prompt in remove_prompts]

        remove_responses = await openai_request(remove_prompts, "")

        if remove_responses:
            print(remove_responses[0])

        for i in range(len(remove_indices)):
            post_remove_items, post_remove_ids, post_remove_intents = bundle_item_remover(remove_items[i], remove_ids[i], remove_responses[i])
            bundle_items[remove_indices[i]] = post_remove_items
            bundle_indices[remove_indices[i]] = post_remove_ids
            # if post_remove_intents is not None:
            #     bundle_intents[remove_indices[i]] = post_remove_intents

        if product_types is not None:

            expand_candidates_prompts = [create_expand_candidates_prompt(add_strings[i], add_summary[i], product_types=product_types[add_indices[i]], domain=domain) for i in range(len(add_indices))]

            if expand_candidates_prompts:

                print(expand_candidates_prompts[0])

            expand_candidates_prompts = [{"prompts": expand_candidates_prompt} for expand_candidates_prompt in expand_candidates_prompts]

            expand_candidates_responses = await openai_request(expand_candidates_prompts, "")

            expanded_items, expanded_ids = extended_candidates_extractor(expand_candidates_responses, domain, num_candidates=2)

            candidate_strings = []

            for i in range(len(add_indices)):

                if session_items_ids is not None:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]], expanded_items[i], expanded_ids[i])

                else:

                    candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], expanded_items[i], expanded_ids[i])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt_no_graphs(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            if add_prompts:

                print(add_prompts[0])

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")


            if add_responses:

                print(add_responses[0])

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids

        else:
            candidate_strings = []

            for i in range(len(add_indices)):

                candidate_items, candidate_ids = extended_candidates_mixer(add_items[i], session_item_titles[add_indices[i]], session_items_ids[add_indices[i]])

                candidate_strings.append(candidate_token_strings(candidate_items, candidate_ids, domain))

            add_prompts = [create_add_item_prompt_no_graphs(add_strings[i], add_summary[i], candidate_strings[i]) for i in range(len(add_indices))]

            add_prompts = [{"prompts": add_prompt} for add_prompt in add_prompts]

            add_responses = await openai_request(add_prompts, "")

            for i in range(len(add_indices)):
                post_add_intent, post_add_items, post_add_ids = bundle_item_adder(add_intents[i], add_items[i], add_ids[i], add_responses[i], domain)
                # bundle_intents[add_indices[i]] = post_add_intent
                bundle_items[add_indices[i]] = post_add_items
                bundle_indices[add_indices[i]] = post_add_ids


        working_indices = [index for index, flag in enumerate(flags) if flag]


        modified_intents = [bundle_intents[i] for i in working_indices]
        modified_items = [bundle_items[i] for i in working_indices]
        modified_ids = [bundle_indices[i] for i in working_indices]

        modified_input_strings = input_strings(modified_intents, modified_items)

        modify, retain, adjusted_scores, summary_responses, adjusted_min_scores = await evaluation_module(charizards=[f"iterative_workflow_bad_{char}_{num_iter + 1}", f"iterative_workflow_middle_{char}_{num_iter + 1}", f"iterative_workflow_good_{char}_{num_iter + 1}"],
                                                      prompts=prompts,
                                                      input_strings=modified_input_strings, score_acceptability = score_acceptability)

        for i in range(len(adjusted_scores)):
            all_scores[working_indices[i]] = adjusted_scores[i]
            min_scores[working_indices[i]] = adjusted_min_scores[i]

        for i in retain:
            flags[working_indices[i]] = False

        working_indices = [index for index, flag in enumerate(flags) if flag]

        for i in range(len(bundle_items)):
            if len(bundle_items[i]) < 2:
                flags[i] = True

        historical_intents.append(copy.deepcopy(bundle_intents))
        historical_bundle_items.append(copy.deepcopy(bundle_items))
        historical_bundle_indices.append(copy.deepcopy(bundle_indices))
        historical_bundle_scores.append(copy.deepcopy(all_scores))
        historical_min_scores.append(copy.deepcopy(min_scores))
        historical_flags.append(copy.deepcopy(flags))


        with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}_{num_iter}.pkl", "wb") as f:
            pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    with open(f"/content/drive/MyDrive/Bundle_Refinement/historical_bundle_changes_{domain}_{char}.pkl", "wb") as f:
        pickle.dump([historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags], f)


    return historical_intents, historical_bundle_items, historical_bundle_indices, historical_bundle_scores, historical_min_scores, historical_flags

In [ ]:
electronic_bundle_intents, electronic_bundle_item_ids, electronic_bundle_item_titles, electronic_session_item_ids, electronic_session_item_titles = [], [], [], [], []

for i in range(len(electronic_bundles_items)):

    bundle_intent, bundle_item_ids, bundle_item_titles, session_item_id, session_item_title = bundle_info_provider(i, "electronic")

    electronic_bundle_intents.append(bundle_intent)
    electronic_bundle_item_ids.append(bundle_item_ids)
    electronic_bundle_item_titles.append(bundle_item_titles)

    electronic_session_item_ids.append(session_item_id)
    electronic_session_item_titles.append(session_item_title)

electronic_starting_product_list = [making_product_type_list(electronic_category_indices[i], all_electronic_products) for i in range(len(electronic_bundles_items))]

clothing_bundle_intents, clothing_bundle_item_ids, clothing_bundle_item_titles, clothing_session_item_ids, clothing_session_item_titles = [], [], [], [], []

for i in range(len(clothing_bundles_items)):

    bundle_intent, bundle_item_ids, bundle_item_titles, session_item_id, session_item_title = bundle_info_provider(i, "clothing")

    clothing_bundle_intents.append(bundle_intent)
    clothing_bundle_item_ids.append(bundle_item_ids)
    clothing_bundle_item_titles.append(bundle_item_titles)

    clothing_session_item_ids.append(session_item_id)
    clothing_session_item_titles.append(session_item_title)

clothing_starting_product_list = [making_product_type_list(clothing_category_indices[i], all_clothing_products) for i in range(len(clothing_bundles_items))]

food_bundle_intents, food_bundle_item_ids, food_bundle_item_titles, food_session_item_ids, food_session_item_titles = [], [], [], [], []

for i in range(len(food_bundles_items)):

    bundle_intent, bundle_item_ids, bundle_item_titles, session_item_id, session_item_title = bundle_info_provider(i, "food")

    food_bundle_intents.append(bundle_intent)
    food_bundle_item_ids.append(bundle_item_ids)
    food_bundle_item_titles.append(bundle_item_titles)

    food_session_item_ids.append(session_item_id)
    food_session_item_titles.append(session_item_title)

food_starting_product_list = [making_product_type_list(food_category_indices[i], all_food_products) for i in range(len(food_bundles_items))]

num_iter = 10 # I did 20 for my runs. You really should do 5 or 10 for the No feedback (no help) workflow, as it exhibits degenerate behaviour


print(len(food_starting_product_list))
print(len(clothing_starting_product_list))
print(len(electronic_starting_product_list))

In [ ]:

# print("electronic_session+expanded_more_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "electronic",
#           char = "complete_electronic_session_expanded_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = electronic_session_item_ids,
#           session_item_titles = electronic_session_item_titles,
#           starting_product_list = electronic_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

# print("clothing_session+expanded_more_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "clothing",
#           char = "complete_clothing_session_expanded_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = clothing_session_item_ids,
#           session_item_titles = clothing_session_item_titles,
#           starting_product_list = clothing_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

# print("food_session+expanded_more_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "food",
#           char = "complete_food_session_expanded_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = food_session_item_ids,
#           session_item_titles = food_session_item_titles,
#           starting_product_list = food_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)




# print("electronic_session_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "electronic",
#           char = "complete_electronic_session_only_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = electronic_session_item_ids,
#           session_item_titles = electronic_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)

# print("clothing_session_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "clothing",
#           char = "complete_clothing_session_only_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = clothing_session_item_ids,
#           session_item_titles = clothing_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)

# print("food_session_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "food",
#           char = "complete_food_session_only_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = food_session_item_ids,
#           session_item_titles = food_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)




# print("electronic_expanded_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "electronic",
#           char = "complete_electronic_expanded_only_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = None,
#           session_item_titles = None,
#           starting_product_list = electronic_starting_product_list,
#           randomise = "",
          # num_iterations=num_iter)

# print("clothing_expanded_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "clothing",
#           char = "complete_clothing_expanded_only_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = None,
#           session_item_titles = None,
#           starting_product_list = clothing_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

# print("food_expanded_only_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await drugs_workflow(domain = "food",
#           char = "complete_food_expanded_only_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = None,
#           session_item_titles = None,
#           starting_product_list = food_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)




# print("electronic_no_enriched_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_enrichment_workflow(domain = "electronic",
#           char = "complete_electronic_no_enrichment_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = electronic_session_item_ids,
#           session_item_titles = electronic_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)

# print("clothing_no_enriched_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_enrichment_workflow(domain = "clothing",
#           char = "complete_clothing_no_enrichment_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = clothing_session_item_ids,
#           session_item_titles = clothing_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)

# print("food_no_enriched_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_enrichment_workflow(domain = "food",
#           char = "complete_food_no_enrichment_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = food_session_item_ids,
#           session_item_titles = food_session_item_titles,
#           randomise = "",
#           num_iterations=num_iter)




# print("electronic_no_help_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_help_workflow(domain = "electronic",
#           char = "complete_electronic_no_evaluator_help_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = electronic_session_item_ids,
#           session_item_titles = electronic_session_item_titles,
#           starting_product_list = electronic_starting_product_list,
#           randomise = "",
#           num_iterations=5)

# print("clothing_no_help_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_help_workflow(domain = "clothing",
#           char = "complete_clothing_no_evaluator_help_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = clothing_session_item_ids,
#           session_item_titles = clothing_session_item_titles,
#           starting_product_list = clothing_starting_product_list,
#           randomise = "",
#           num_iterations=5)

# print("food_no_help_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_help_workflow(domain = "food",
#           char = "complete_food_no_evaluator_help_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = food_session_item_ids,
#           session_item_titles = food_session_item_titles,
#           starting_product_list = food_starting_product_list,
#           randomise = "",
#           num_iterations=5)




# print("electronic_no_graph_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_graphs_workflow(domain = "electronic",
#           char = "complete_electronic_no_graph_help_run",
#           prompts = [electronic_bad_prompt, electronic_middle_prompt, electronic_good_prompt]
#           , starting_bundle_intents = electronic_bundle_intents,
#           starting_bundle_item_ids = electronic_bundle_item_ids,
#           starting_bundle_item_titles = electronic_bundle_item_titles,
#           session_items_ids = electronic_session_item_ids,
#           session_item_titles = electronic_session_item_titles,
#           starting_product_list = electronic_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

# print("clothing_no_graph_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_graphs_workflow(domain = "clothing",
#           char = "complete_clothing_no_graph_help_run",
#           prompts = [clothing_bad_prompt, clothing_middle_prompt, clothing_good_prompt]
#           , starting_bundle_intents = clothing_bundle_intents,
#           starting_bundle_item_ids = clothing_bundle_item_ids,
#           starting_bundle_item_titles = clothing_bundle_item_titles,
#           session_items_ids = clothing_session_item_ids,
#           session_item_titles = clothing_session_item_titles,
#           starting_product_list = clothing_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

# print("food_no_graph_drugs")
# session_expanded_intents, session_expanded_bundle_items, session_expanded_bundle_indices, session_expanded_bundle_scores, session_expanded_min_scores, session_expanded_flags = await no_graphs_workflow(domain = "food",
#           char = "complete_food_no_graph_help_run",
#           prompts = [food_bad_prompt, food_middle_prompt, food_good_prompt]
#           , starting_bundle_intents = food_bundle_intents,
#           starting_bundle_item_ids = food_bundle_item_ids,
#           starting_bundle_item_titles = food_bundle_item_titles,
#           session_items_ids = food_session_item_ids,
#           session_item_titles = food_session_item_titles,
#           starting_product_list = food_starting_product_list,
#           randomise = "",
#           num_iterations=num_iter)

